# Sim-to-Real Detection with NVIDIA Synthetic Data

Runtime -> Change runtime type -> **GPU (T4)** before running.
This notebook clones the project repo and runs the pipeline end to end.


## 0. Setup


In [ ]:
# If you pushed this project to GitHub, clone it. Otherwise upload the src/ folder.
# !git clone https://github.com/<you>/sim2real-nvidia.git && cd sim2real-nvidia
!pip install -q -r requirements.txt


In [ ]:
# Log in so you can download the dataset (token: huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()


## 1. Explore the dataset layout
Then edit `SYNTH_PATTERNS` / `REAL_PATTERNS` in `src/config.py` to match.


In [ ]:
!python src/explore_repo.py --grep MTMC_Tracking_2026


## 2. Download a SMALL subset first


In [ ]:
!python src/download_data.py --which both


## 3a. Confirm the annotation schema
Point --path at one downloaded ground_truth.json. If keys differ, fix
`parse_ground_truth()` in `src/prepare_yolo.py`.


In [ ]:
!find data/raw -name ground_truth.json | head -3


In [ ]:
# !python src/inspect_annotations.py --path data/raw/synth/<...>/ground_truth.json


## 3b. Build the YOLO datasets


In [ ]:
!python src/prepare_yolo.py --split synth
!python src/prepare_yolo.py --split real


## 4. Train the baseline (synthetic only)


In [ ]:
!python src/train.py --data data/yolo_synth/data.yaml --name synth_only --epochs 50


## 5. Measure the sim-to-real gap


In [ ]:
!python src/evaluate.py --weights runs/detect/synth_only/weights/best.pt --data data/yolo_synth/data.yaml --tag synth_only__on_synth
!python src/evaluate.py --weights runs/detect/synth_only/weights/best.pt --data data/yolo_real/data.yaml --tag synth_only__on_real


## 6. Ablation: fine-tune with a few real frames
Build a mixed set (synthetic + k% real), then fine-tune from the baseline
and re-evaluate. Repeat for k = 1, 5, 10 and plot mAP-on-real vs. k.


In [ ]:
import pandas as pd
pd.read_csv('results/metrics.csv')
